In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
np.random.seed(18)

import tensorflow as tf

from scipy.io import loadmat
import tifffile
import cv2

from func_file_Model import ResNet_model_paper
from func_file_Tubulin import *

## Load data

In [ ]:
#Deep-STORM reconstructed array
ds_reconstructed = loadmat("DS_reconstruction.mat")["Recovery"]

#SOS Plugin localization table
sos_table = np.loadtxt("SOS_detections.txt")

# Measured frames
frames = Load_sequence()
frames_normed, norms = Normalize_data(frames)
frames_normed.shape

## SOSplugin reconstruction

In [ ]:
#Extract emitter information from localization table and render an image
y = sos_table[:, 0]
x = sos_table[:, 1]
intensity = sos_table[:, 3]
stdev_y = sos_table[:, 6]
stdev_x = sos_table[:, 7]

sos_reconstructed = Render_SOS_Plugin(x, y, intensity, stdev_x, stdev_y)

## DAMN reconstruction

Tensorflow may print warnings.

In [ ]:
#Load the model
model_ResNet = ResNet_model_paper(channels = 64, num_blocks_array = [3,3,3,3], 
                                  kernel_sizes=[5,7,9,11], LeakyReLU_slope=0.1, 
                                  padding="same", interpolation="bilinear", kernel_initializer="he_uniform")

#Compile and load weights
_ = model_ResNet(tf.random.normal([1, 50, 50, 1]))
model_ResNet.load_weights("DAMN_upsampling_model.keras")

In [ ]:
#Reconstruct all data using the model. Takes approximately 5 min on our computational resources.

evaluation_batch = 8   #Set the evaluation batch_size depending on your available GPU memory

DAMN_predicted = np.squeeze(model_ResNet.predict(np.expand_dims(frames_normed, axis=-1), batch_size=evaluation_batch, verbose=1))

DAMN_reconstructed = np.mean(DAMN_predicted * norms[:,None,None], axis=0)
DAMN_reconstructed.shape

## Vizualize results

### Full image

In [ ]:
plt.figure(figsize=(30,10))
plt.subplot(141)
plt.matshow(frames.sum(axis=0), cmap="viridis", fignum=False)
plt.title("Low-resolution image")

plt.subplot(142)
plt.matshow(sos_reconstructed, cmap="viridis", fignum=False, vmax=35*sos_reconstructed.mean())
plt.title("SOSplugin reconstruction")

plt.subplot(143)
plt.matshow(ds_reconstructed, cmap="viridis", fignum=False, vmax=35*ds_reconstructed.mean())
plt.title("Deep-STORM reconstruction")

plt.subplot(144)
plt.matshow(DAMN_reconstructed, cmap="viridis", fignum=False, vmax=35*DAMN_reconstructed.mean())
plt.title("DAMN reconstruction")

plt.show()

### Zoom cutout

In [ ]:
a1 = 20          #Position to vertically start at (in the 128x128 field)
a2 = a1 + 30     #Position to vertically end at (in the 128x128 field)

b1 = 20          #Position to horizontally start at (in the 128x128 field)
b2 = b1 + 30     #Position to horizontally end at (in the 128x128 field)

In [ ]:
#Cut approximatelly (up to the origin correction) the same areas from the images
frame_cutout = frames[:,a1:a2,b1:b2].sum(axis=0)
sos_reconstructed_cutout = sos_reconstructed[10*a1:10*a2, 10*b1:10*b2]
ds_reconstructed_cutout = ds_reconstructed[8*a1:8*a2, 8*b1:8*b2]
DAMN_reconstructed_cutout = DAMN_reconstructed[8*a1:8*a2, 8*b1:8*b2]

In [ ]:
plt.figure(figsize=(30,10))
plt.subplot(141)
plt.matshow(frame_cutout, cmap="viridis", fignum=False)
plt.title("High-concentration sample")

plt.subplot(142)
plt.matshow(sos_reconstructed_cutout, cmap="viridis", fignum=False, vmax=20*sos_reconstructed_cutout.mean())
plt.title("SOSplugin reconstruction - zoom")

plt.subplot(143)
plt.matshow(ds_reconstructed_cutout, cmap="viridis", fignum=False, vmax=20*ds_reconstructed_cutout.mean())
plt.title("Deep-STORM reconstruction - zoom")

plt.subplot(144)
plt.matshow(DAMN_reconstructed_cutout, cmap="viridis", fignum=False, vmax=20*DAMN_reconstructed_cutout.mean())
plt.title("DAMN reconstruction - zoom")

plt.show()

### Microtubule profile

#### Zoom and correct tubulin orientation

In [ ]:
c1 = 45          #Position to vertically start at (in the 128x128 field)
c2 = c1 + 30     #Position to vertically end at (in the 128x128 field)

d1 = 35          #Position to horizontally start at (in the 128x128 field)
d2 = d1 + 30     #Position to horizontally end at (in the 128x128 field)

In [ ]:
#Cut approximatelly (up to the origin correction) the same areas from the images
sos_reconstructed_MP = sos_reconstructed[10*c1:10*c2, 10*d1:10*d2]
ds_reconstructed_MP = ds_reconstructed[8*c1:8*c2, 8*d1:8*d2]
DAMN_reconstructed_MP = DAMN_reconstructed[8*c1:8*c2, 8*d1:8*d2]

#Correct the tubulin orientation
sos_corrected = Apply_correction(sos_reconstructed_MP, [300, 360, 30, 37.5, 55])
ds_corrected = Apply_correction(ds_reconstructed_MP, [240, 300, 30, 30, 50])
DAMN_corrected = Apply_correction(DAMN_reconstructed_MP, [240, 300, 30, 30, 50])

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(131)
plt.matshow(sos_corrected, cmap="viridis", fignum=False, vmax=50*sos_corrected.mean())
plt.title("SOSplugin")

plt.subplot(132)
plt.matshow(ds_corrected, cmap="viridis", fignum=False, vmax=50*ds_corrected.mean())
plt.title("Deep-STORM")

plt.subplot(133)
plt.matshow(DAMN_corrected, cmap="viridis", fignum=False, vmax=50*DAMN_corrected.mean())
plt.title("DAMN")

plt.show()

#### Profile

In [ ]:
#Choose the area for profile estimation and project over the vertical axis
sos_to_profile = sos_corrected[62:-63, 93:-94]
ds_to_profile = ds_corrected[50:-50, 75:-75]
DAMN_to_profile = DAMN_corrected[50:-50, 75:-75]

sos_profile = sos_to_profile.sum(axis=0)
ds_profile = ds_to_profile.sum(axis=0)
DAMN_profile = DAMN_to_profile.sum(axis=0)

sos_profile = sos_profile / sos_profile.max()
ds_profile = ds_profile / ds_profile.max()
DAMN_profile = DAMN_profile / DAMN_profile.max()

In [ ]:
#Horizontal axes information
sos_pixel_size  = 10.0 #nm
ds_pixel_size   = 12.5 #nm
damn_pixel_size = 12.5 #nm

sos_axis = np.linspace(0, sos_profile.shape[0]-1, sos_profile.shape[0]) * sos_pixel_size
ds_axis = np.linspace(0, ds_profile.shape[0]-1, ds_profile.shape[0]) * ds_pixel_size
DAMN_axis = np.linspace(0, DAMN_profile.shape[0]-1, DAMN_profile.shape[0]) * damn_pixel_size

In [ ]:
#Plot profiles
plt.figure(figsize=(15,5))
plt.subplot(131)
plt.plot(sos_axis, sos_profile, color="green")
plt.title("SOSplugin")
plt.xlabel("Position [nm]")

plt.subplot(132)
plt.plot(ds_axis, ds_profile, color="purple")
plt.title("Deep-STORM")
plt.xlabel("Position [nm]")

plt.subplot(133)
plt.plot(DAMN_axis, DAMN_profile, color="red")
plt.title("DAMN")
plt.xlabel("Position [nm]")

plt.show()

In [ ]:
#Make the profiles overlap at the center
sos_profile_cut = sos_profile[15-2:45-2]
ds_profile_cut = ds_profile[12:36]
DAMN_profile_cut = DAMN_profile[12+1:36+1]

sos_cut_axis = np.linspace(0, sos_profile_cut.shape[0]-1, sos_profile_cut.shape[0]) * sos_pixel_size
ds_cut_axis = np.linspace(0, ds_profile_cut.shape[0]-1, ds_profile_cut.shape[0]) * ds_pixel_size
DAMN_cut_axis = np.linspace(0, DAMN_profile_cut.shape[0]-1, DAMN_profile_cut.shape[0]) * damn_pixel_size

In [ ]:
#Plot overlapping profiles
plt.figure(figsize=(8,5))
plt.plot(sos_cut_axis, sos_profile_cut, color="green", label="SOSplugin")
plt.fill_between(sos_cut_axis, sos_profile_cut, 0, color="green", alpha=0.1)

plt.plot(ds_cut_axis, ds_profile_cut, color="purple", label="Deep-STORM")
plt.fill_between(ds_cut_axis, ds_profile_cut, 0, color="purple", alpha=0.1)

plt.plot(DAMN_cut_axis, DAMN_profile_cut, color="red", label="DAMN")
plt.fill_between(DAMN_cut_axis, DAMN_profile_cut, 0, color="red", alpha=0.1)

plt.plot(sos_cut_axis, np.ones(sos_cut_axis.shape)/2, color="black", linestyle="--")

plt.xlabel("Position [nm]")
plt.legend()

plt.show()

In [ ]:
#Interpolate the widths
print("SOS profile FWHM: ", np.round(FWHM_interpolate(sos_profile_cut, 10), 2), "nm")
print("DS profile FWHM:  ", np.round(FWHM_interpolate(ds_profile_cut, 12.5), 2), "nm")
print("DAMN profile FWHM:", np.round(FWHM_interpolate(DAMN_profile_cut, 12.5), 2), "nm")